<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session06-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 6 lab: PyMOL, structure comparison and structure search {.unnumbered}

This lab follows the Session 6 lab quiz on Canvas, section by section
(A-F). Run it on Colab, in the browser or from VS Code with the Google
Colab extension (see the Setup page). You do **not** need to install
PyMOL on your own computer: the first cell installs open-source PyMOL as
a Python library on the Colab machine.

**How PyMOL works here.** In the desktop program you type commands such
as `color red, 7LXM and ss H` at the `PyMOL>` prompt. Here you type the
same commands inside `cmd.do("...")`, one command per line. PyMOL has no
window on Colab, so to *see* the structure you call `snapshot()`, which
draws the current view as a picture. Every PyMOL command also exists as
a Python function (`cmd.fetch`, `cmd.color`, `cmd.select`,
`cmd.cealign`, ...), which is how the questions that need a number get
it.

| Part | What | Quiz section |
|---|---|---|
| 1 | Install and set up PyMOL | - |
| 2 | An antibody-antigen complex: representations, chains, secondary structure | A |
| 3 | Copies of one chain, and a structural alignment | B, C |
| 4 | A chitinase with bound metal ions | D |
| 5 | Neuroglobin: TM-align against three structures | E |
| 6 | Neuroglobin: a Foldseek search of the whole PDB | E |
| 7 | Structural domains in CATH | F |

## 1. Install and set up PyMOL

In [ ]:
# Install the libraries Colab does not come with (does nothing if they are already installed).
# On PyPI, open-source PyMOL is the package "pymol-open-source-whl"; it is imported as "pymol".
import importlib.util, subprocess, sys
for module, package in [("pymol", "pymol-open-source-whl"), ("tmtools", "tmtools"),
                        ("py3Dmol", "py3Dmol"), ("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
from pymol import cmd
from IPython.display import Image, display

cmd.bg_color("white")
cmd.set("ray_opaque_background", 1)

def snapshot(width=640, height=480):
    # Ray-trace the current PyMOL view and show it below the cell (takes a few seconds).
    cmd.png("_view.png", width=width, height=height, ray=1)
    display(Image("_view.png"))

print("PyMOL", cmd.get_version()[0], "is ready")

## 2. An antibody-antigen complex (quiz section A)

`7LXM` is a cryo-EM structure of the HIV envelope glycoprotein trimer
bound by a broadly neutralising antibody, PGT122. Fetch it in PDB format
(`type=pdb`): PyMOL's default mmCIF download gives individual sugars and
waters their own chain letters, which would inflate the chain count.
Read the `COMPND` lines that PyMOL prints: they say what each chain is.

In [ ]:
cmd.do('''
fetch 7LXM, type=pdb, async=0
hide everything
show cartoon, polymer
color grey60, 7LXM
color red, 7LXM and ss H
color yellow, 7LXM and ss S
color blue, 7LXM and ss L+''
orient 7LXM
''')
snapshot()

Red helices, yellow strands, blue loops. Now colour by chain instead
(`util.cbc`, "colour by chain") and look at the picture again: can you
see the threefold symmetry of the trimer?

In [ ]:
cmd.do("util.cbc 7LXM and polymer")
snapshot()

How many protein chains are there, and how many *different* sequences?
`polymer.protein` restricts the selection to protein chains (no sugars,
no water).

In [ ]:
chains = cmd.get_chains("7LXM and polymer.protein")
print(len(chains), "protein chains:", chains)

sequences = {}
for c in chains:
    fasta = cmd.get_fastastr(f"7LXM and polymer.protein and chain {c}")
    sequences[c] = "".join(fasta.split("\n")[1:])
print(len(set(sequences.values())), "unique sequences")
for seq in set(sequences.values()):
    print("  chains", [c for c in chains if sequences[c] == seq], f"({len(seq)} residues)")

Which secondary-structure states does PyMOL's own assignment (`dss`)
find? The letters are `H` (helix), `S` (strand) and `L` (loop).

In [ ]:
from collections import Counter

cmd.dss("7LXM")
states = []
cmd.iterate("7LXM and polymer.protein and name CA", "states.append(ss)", space={"states": states})
print(Counter(states))

**Optional: rotate it yourself.** `snapshot()` gives a still picture.
For an interactive view you can drag with the mouse, hand the
coordinates to py3Dmol. It works in Colab in the browser; in VS Code,
whether the viewer appears depends on your setup, so skip it if nothing
shows.

In [ ]:
import py3Dmol

view = py3Dmol.view(width=640, height=480)
view.addModel(cmd.get_pdbstr("7LXM and polymer.protein"), "pdb")
view.setStyle({"cartoon": {"colorscheme": "chain"}})
view.zoomTo()
view.show()

## 3. Copies of one chain, and a structural alignment (quiz sections B and C)

Chains H, N and P are the three copies of the antibody's heavy chain, one
per protomer of the trimer. Copy them into three separate objects, then
superimpose two of them with `cealign`, which reports the RMSD after the
best superposition.

In [ ]:
cmd.do('''
color grey60, 7LXM
create obj_H, 7LXM and chain H
create obj_N, 7LXM and chain N
create obj_P, 7LXM and chain P
''')
result = cmd.cealign("obj_H", "obj_N")
print(f"RMSD {result['RMSD']:.2f} Å over {result['alignment_length']} aligned residues")

In [ ]:
cmd.do('''
hide everything
show cartoon, obj_H or obj_N
color skyblue, obj_H
color salmon, obj_N
orient obj_H or obj_N
''')
snapshot()

## 4. A chitinase with bound metal ions (quiz section D)

`8C6Z` is a bacterial chitinase with two kinds of bound metal ion and a
small-molecule inhibitor. Show the charged residues as red sticks, then
find out which ions are bound (`inorganic` selects ions) and which
residues surround them.

In [ ]:
cmd.do('''
delete all
fetch 8C6Z, type=pdb, async=0
hide everything
show cartoon, polymer
color grey80, polymer
select charged, resn ASP+GLU+LYS+ARG+HIS
show sticks, charged and not name N+C+O
color red, charged
show spheres, inorganic
orient 8C6Z
''')
snapshot()

ions = set()
cmd.iterate("inorganic", "ions.add(resn)", space={"ions": ions})
print("Bound ions:", ions)

In [ ]:
near = set()
cmd.iterate("polymer and byres (all within 3 of inorganic)", "near.add((resn, int(resi), chain))",
            space={"near": near})
for resn, resi, chain in sorted(near, key=lambda r: r[1]):
    print(resn, resi, "chain", chain)

cmd.do('''
hide everything
show sticks, polymer and byres (all within 3 of inorganic)
show spheres, inorganic
color orange, inorganic
util.cbaw polymer
orient inorganic
zoom inorganic, 6
''')
snapshot()

The close-up is coloured by element (`util.cbaw`): carbon white, oxygen
red, nitrogen blue, sulfur yellow. Which atoms point at the metal ions?

## 5. Neuroglobin: one structure against three others (quiz section E)

`1OJ6` is the X-ray structure of human neuroglobin. We compare its chain
A with three structures, from "the same" to "unrelated":

1. neuroglobin's **AlphaFold DB prediction** (UniProt `Q9NPG2`): the same
   protein, by an independent method;
2. **`2HHB`**, human haemoglobin: a different globin;
3. **`1EMA`**, green fluorescent protein: an unrelated fold.

TM-align (through the `tmtools` package) finds the superposition that
maximises the TM-score; we normalise by the length of neuroglobin.

In [ ]:
import requests
from tmtools import tm_align
from tmtools.io import get_structure, get_residue_data

for pdb_id in ["1OJ6", "2HHB", "1EMA"]:
    r = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30)
    r.raise_for_status()
    open(f"{pdb_id}.pdb", "w").write(r.text)

# The current AlphaFold DB model for neuroglobin: ask the API for its file name
# rather than hard-coding a version that will be superseded.
af = requests.get("https://alphafold.ebi.ac.uk/api/prediction/Q9NPG2", timeout=30).json()[0]
open("AF-Q9NPG2.pdb", "w").write(requests.get(af["pdbUrl"], timeout=30).text)
print("AlphaFold DB model version", af["latestVersion"])

def chain(path, chain_id):
    # (CA coordinates, sequence) of one chain
    for c in get_structure(path).get_chains():
        if c.id == chain_id:
            return get_residue_data(c)

ngb_xyz, ngb_seq = chain("1OJ6.pdb", "A")
print(f"\n{'1OJ6 chain A compared with':<34}{'TM-score':>9}{'RMSD':>7}{'seq id':>8}")
for name, path, chain_id in [("AlphaFold DB model of Q9NPG2", "AF-Q9NPG2.pdb", "A"),
                             ("2HHB chain A (alpha-globin)", "2HHB.pdb", "A"),
                             ("2HHB chain B (beta-globin)", "2HHB.pdb", "B"),
                             ("1EMA chain A (GFP)", "1EMA.pdb", "A")]:
    xyz, seq = chain(path, chain_id)
    res = tm_align(ngb_xyz, xyz, ngb_seq, seq)
    pairs = [(a, b) for a, b in zip(res.seqxA, res.seqyA) if a != "-" and b != "-"]
    ident = sum(a == b for a, b in pairs) / len(pairs)
    print(f"{name:<34}{res.tm_norm_chain1:>9.2f}{res.rmsd:>7.1f}{ident:>8.0%}")

Look at the haemoglobin rows: about one residue in five is identical,
yet the TM-score is about 0.8. Then superimpose neuroglobin and
haemoglobin's β chain in PyMOL to see it.

In [ ]:
cmd.do('''
delete all
load 1OJ6.pdb, ngb
load 2HHB.pdb, hb
hide everything
show cartoon, (ngb and chain A) or (hb and chain B)
color skyblue, ngb
color salmon, hb
''')
result = cmd.cealign("ngb and chain A", "hb and chain B")
print(f"cealign: RMSD {result['RMSD']:.1f} Å over {result['alignment_length']} residues")
cmd.orient("ngb and chain A")
snapshot()

## 6. Searching the whole PDB by structure with Foldseek (quiz section E)

TM-align compares two structures you choose. **Foldseek** *searches*:
it turns every structure into a string of 3Di letters and compares
those strings as fast as MMseqs2 compares sequences (see the book page).
Here we send neuroglobin's chain A to the Foldseek web server and search
`pdb100`, every structure in the PDB. This is the same search as on
[search.foldseek.com](https://search.foldseek.com/search), done through
its API. It usually takes under a minute; if the server is busy, run the
cell again later.

In [ ]:
import time

atoms = [line for line in open("1OJ6.pdb") if line.startswith("ATOM") and line[21] == "A"]
ticket = requests.post("https://search.foldseek.com/api/ticket",
                       files={"q": ("1OJ6_A.pdb", "".join(atoms) + "END\n")},
                       data=[("mode", "3diaa"), ("database[]", "pdb100")], timeout=60).json()
print("submitted, ticket", ticket["id"])

for _ in range(60):
    status = requests.get(f"https://search.foldseek.com/api/ticket/{ticket['id']}", timeout=30).json()["status"]
    if status in ("COMPLETE", "ERROR"):
        break
    time.sleep(5)
print("status:", status)

result = requests.get(f"https://search.foldseek.com/api/result/{ticket['id']}/0", timeout=60).json()
hits = result["results"][0]["alignments"][0]
print(len(hits), "hits in the PDB")

In [ ]:
print(f"{'#':>3}  {'target':<62}{'seq id':>7}{'E-value':>10}")
for i, h in enumerate(hits[:40], 1):
    print(f"{i:>3}  {h['target'][:60]:<62}{h['seqId']:>6.0f}%{h['eval']:>10.0e}")

The first hits are neuroglobins: human and mouse, wild type and mutants,
with 85-100% identity. Read further down the list. What kinds of protein
come next, and what is their sequence identity to neuroglobin? Are the
hits still significant?

## 7. Structural domains in CATH (quiz section F)

Sessions 3 and 5 classified domains by *sequence* (InterPro, Pfam).
CATH classifies them by 3D shape. PDBe's API maps a PDB entry to its
CATH domains.

In [ ]:
cath = requests.get("https://www.ebi.ac.uk/pdbe/api/mappings/cath/1oj6", timeout=30).json()
for cath_id, entry in cath["1oj6"]["CATH"].items():
    print(cath_id, "|", "class:", entry["class"], "| architecture:", entry["architecture"],
          "| topology:", entry["topology"], "| homology:", entry["homology"])

In [ ]:
# Clean up the files this notebook wrote
import glob, os
for f in ["_view.png", "AF-Q9NPG2.pdb"] + glob.glob("????.pdb"):
    if os.path.exists(f):
        os.remove(f)

## Done

Read the TM-scores of part 5 as one story:

```
same protein, experimental vs. AlphaFold prediction  → nearly identical
a different globin (haemoglobin)                     → same fold, about 20% sequence identity
GFP                                                  → unrelated fold
```

The middle line is the central lesson of Session 6: structure is more
conserved than sequence, and part 6 shows a structure search using it
to find relatives across the whole PDB. The first line previews Session
12: here AlphaFold DB simply supplied a structure, and Session 12 asks
how that prediction is made and which parts of it to trust.

Answer the Session 6 lab quiz on Canvas with the numbers from your own
run.

**Optional: a viewer inside VS Code.** For a quick look at a structure
without writing code, the **Protein Viewer** extension for VS Code
(`ArianJamasb.protein-viewer`, built on Mol*, the viewer that RCSB and
PDBe use) opens any PDB entry: Command Palette → "Start Protein Viewer",
then type an accession such as `8C6Z`.